In [1]:
import os
import sys

sys.path.append('/usr/lib/python3/site-packages')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/soft/DPU-PYNQ')

from pynq_dpu import DpuOverlay

WORK_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae'
PL_DIR = os.path.join(WORK_DIR, 'zcu111_1920x1080')

BIT_PATH = os.path.join(PL_DIR, 'dpu.bit')

overlay = DpuOverlay(BIT_PATH)

print("Overlay loaded OK")
print("IPs:")
for k in overlay.ip_dict.keys():
    print(" ", k)

if hasattr(overlay, "vq_accel_1"):
    vq_accel = overlay.vq_accel_1
elif hasattr(overlay, "vq_accel_0"):
    vq_accel = overlay.vq_accel_0
else:
    raise RuntimeError("Cannot find vq_accel_1 or vq_accel_0")

print(vq_accel.register_map)

Overlay loaded OK
IPs:
  axi_intc_0
  DPUCZDX8G_1
  vq_accel_1
  vq_dequant_1
  ps_e
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, AP_CONTINUE=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  in_z_1 = Register(in_z=write-only),
  in_z_2 = Register(in_z=write-only),
  in_codebook_1 = Register(in_codebook=write-only),
  in_codebook_2 = Register(in_codebook=write-only),
  out_idx_1 = Register(out_idx=write-only),
  out_idx_2 = Register(out_idx=write-only),
  enc_scale = Register(enc_scale=write-only),
  dec_scale_inv = Register(dec_scale_inv=write-only)
}


In [2]:
import os
import sys
import time
import struct
import numpy as np
from pynq import allocate

sys.path.append('/usr/lib/python3/site-packages')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/soft/DPU-PYNQ')

import vart
import xir

WORK_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae'

PRE_DIR = os.path.join(WORK_DIR, 'imgs_preprocessed')
CODEBOOK_PATH = os.path.join(WORK_DIR, 'codebook.npy')
ENC_XMODEL = os.path.join(WORK_DIR, 'xmodel/encoder_1920x1080_som.xmodel')

SAVE_DIR = os.path.join(WORK_DIR, 'debug_1920_ps_dequant')
IDX_DIR = os.path.join(SAVE_DIR, 'idx_bins')
ZQ_DIR = os.path.join(SAVE_DIR, 'zq_npy')

os.makedirs(IDX_DIR, exist_ok=True)
os.makedirs(ZQ_DIR, exist_ok=True)

TARGET_W, TARGET_H = 1920, 1080
LATENT_W = 480
LATENT_H = 270

num_vectors = LATENT_H * LATENT_W
dim = 64
num_code = 512

enc_in_scale = 0.015625
enc_out_scale = 0.015625
dec_in_scale = 0.03125
dec_scale_inv = 1.0 / dec_in_scale

expected_enc_in = (1, TARGET_H, TARGET_W, 3)
expected_enc_out = (1, LATENT_H, LATENT_W, dim)


def set_u64(mmio, lo_off, hi_off, addr):
    mmio.write(lo_off, addr & 0xFFFFFFFF)
    mmio.write(hi_off, (addr >> 32) & 0xFFFFFFFF)


def write_float(mmio, off, value):
    mmio.write(off, struct.unpack('<I', struct.pack('<f', np.float32(value)))[0])


def start_and_wait_old_style(mmio, timeout_s=30.0):
    mmio.write(0x00, 0x11)

    t0 = time.time()
    while (mmio.read(0x00) & 0x02) == 0:
        if time.time() - t0 > timeout_s:
            ctrl = mmio.read(0x00)
            raise RuntimeError(f"IP timeout waiting for AP_DONE, CTRL=0x{ctrl:08X}")
        time.sleep(0.0001)


def get_dpu_subgraph(path):
    graph = xir.Graph.deserialize(path)
    root = graph.get_root_subgraph()
    children = root.toposort_child_subgraph()

    dpu_subgraphs = [
        s for s in children
        if s.has_attr("device") and s.get_attr("device").upper() == "DPU"
    ]

    if len(dpu_subgraphs) == 0:
        raise RuntimeError("No DPU subgraph found")

    return graph, dpu_subgraphs[0]


def ps_dequant_blocked(idx_res, codebook, out_zq_flat, dec_scale_inv, block_vectors=2048):
    idx_res = idx_res.astype(np.uint16, copy=False)

    idx_min = int(idx_res.min())
    idx_max = int(idx_res.max())

    if idx_min < 0 or idx_max >= codebook.shape[0]:
        raise ValueError(f"Invalid idx range: min={idx_min}, max={idx_max}")

    n = idx_res.shape[0]

    for s in range(0, n, block_vectors):
        e = min(s + block_vectors, n)

        tmp = codebook[idx_res[s:e]].astype(np.float32, copy=True)
        tmp *= np.float32(dec_scale_inv)

        np.rint(tmp, out=tmp)
        np.clip(tmp, -128, 127, out=tmp)

        out_zq_flat[s:e, :] = tmp.astype(np.int8)

    return out_zq_flat


if "overlay" not in globals():
    raise RuntimeError("请先运行 Cell A 加载 overlay")

if "vq_accel" not in globals():
    vq_accel = overlay.vq_accel_1

print("Create encoder runner only...")
_, enc_subgraph = get_dpu_subgraph(ENC_XMODEL)
enc_runner = vart.Runner.create_runner(enc_subgraph, "run")

enc_in_tensors = enc_runner.get_input_tensors()
enc_out_tensors = enc_runner.get_output_tensors()

print("Encoder input :", tuple(enc_in_tensors[0].dims))
print("Encoder output:", tuple(enc_out_tensors[0].dims))

if tuple(enc_in_tensors[0].dims) != expected_enc_in:
    raise ValueError("Encoder input shape mismatch")

if tuple(enc_out_tensors[0].dims) != expected_enc_out:
    raise ValueError("Encoder output shape mismatch")

codebook = np.load(CODEBOOK_PATH).astype(np.float32)
assert codebook.shape == (num_code, dim)

vq_codebook_buf = allocate(shape=(num_code, dim), dtype=np.float32, cacheable=1)
vq_codebook_buf[:] = codebook
vq_codebook_buf.sync_to_device()

vq_in_buf = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq_idx_buf = allocate(shape=(num_vectors,), dtype=np.uint16, cacheable=1)

set_u64(vq_accel.mmio, 0x1C, 0x20, vq_codebook_buf.device_address)
write_float(vq_accel.mmio, 0x34, enc_out_scale)
write_float(vq_accel.mmio, 0x3C, dec_scale_inv)

data_files = sorted([f for f in os.listdir(PRE_DIR) if f.endswith('.npy')])

if len(data_files) == 0:
    raise RuntimeError(f"No .npy files found in {PRE_DIR}")

print("num images:", len(data_files))

for img_id, f in enumerate(data_files):
    path = os.path.join(PRE_DIR, f)
    data = np.load(path)

    if data.shape == (TARGET_H, TARGET_W, 3):
        pass
    elif data.shape == (1, TARGET_H, TARGET_W, 3):
        data = data[0]
    else:
        raise ValueError(f"Input shape mismatch: {data.shape}")

    if data.dtype == np.int8:
        enc_input = np.ascontiguousarray(data[np.newaxis])
    else:
        enc_input = np.clip(
            np.round(data.astype(np.float32) / enc_in_scale),
            -128,
            127
        ).astype(np.int8)[np.newaxis]
        enc_input = np.ascontiguousarray(enc_input)

    enc_out = np.empty(expected_enc_out, dtype=np.int8, order='C')

    print(f"\n[Encoder] img={img_id}")
    t0 = time.time()
    job_id = enc_runner.execute_async([enc_input], [enc_out])
    enc_runner.wait(job_id)
    enc_ms = (time.time() - t0) * 1000
    print(f"  encoder time = {enc_ms:.2f} ms")

    vq_in_buf[:] = enc_out.reshape(num_vectors, dim)
    vq_in_buf.sync_to_device()

    vq_idx_buf[:] = 0
    vq_idx_buf.sync_to_device()

    set_u64(vq_accel.mmio, 0x10, 0x14, vq_in_buf.device_address)
    set_u64(vq_accel.mmio, 0x28, 0x2C, vq_idx_buf.device_address)

    print(f"[vq_accel] img={img_id}")
    t0 = time.time()
    start_and_wait_old_style(vq_accel.mmio, timeout_s=30.0)
    vq_ms = (time.time() - t0) * 1000
    print(f"  vq time = {vq_ms:.2f} ms")

    vq_idx_buf.sync_from_device()
    idx_hw = np.array(vq_idx_buf, dtype=np.uint16, copy=True)

    idx_min = int(idx_hw.min())
    idx_max = int(idx_hw.max())
    print(f"  idx range = [{idx_min}, {idx_max}]")

    if idx_max >= num_code:
        raise RuntimeError("Invalid index")

    idx_path = os.path.join(IDX_DIR, f"idx_{img_id:04d}.bin")
    with open(idx_path, "wb") as fp:
        fp.write(idx_hw.tobytes())

    zq_flat = np.empty((num_vectors, dim), dtype=np.int8, order='C')

    print(f"[PS dequant] img={img_id}")
    t0 = time.time()
    ps_dequant_blocked(idx_hw, codebook, zq_flat, dec_scale_inv, block_vectors=2048)
    deq_ms = (time.time() - t0) * 1000

    print(f"  dequant time = {deq_ms:.2f} ms")
    print(f"  zq range = [{int(zq_flat.min())}, {int(zq_flat.max())}]")

    zq_path = os.path.join(ZQ_DIR, f"zq_{img_id:04d}.npy")
    np.save(zq_path, zq_flat.reshape(1, LATENT_H, LATENT_W, dim))

    print("  saved:", idx_path)
    print("  saved:", zq_path)

del enc_runner
print("Encoder runner released.")
print("Done. Decoder was not created.")

Create encoder runner only...
Encoder input : (1, 1080, 1920, 3)
Encoder output: (1, 270, 480, 64)
num images: 10

[Encoder] img=0
  encoder time = 115.83 ms
[vq_accel] img=0
  vq time = 70.85 ms
  idx range = [6, 511]
[PS dequant] img=0
  dequant time = 217.81 ms
  zq range = [-60, 70]
  saved: /home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/debug_1920_ps_dequant/idx_bins/idx_0000.bin
  saved: /home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/debug_1920_ps_dequant/zq_npy/zq_0000.npy

[Encoder] img=1
  encoder time = 113.60 ms
[vq_accel] img=1
  vq time = 70.77 ms
  idx range = [6, 511]
[PS dequant] img=1
  dequant time = 222.63 ms
  zq range = [-60, 70]
  saved: /home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/debug_1920_ps_dequant/idx_bins/idx_0001.bin
  saved: /home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/debug_1920_ps_dequant/zq_npy/zq_0001.npy

[Encoder] img=2
  encoder time = 113.54 ms
[vq_accel] img=2
  vq time = 70.77 ms
  idx range = [6, 511]
[PS dequant] img=2
  dequant time = 217